# 05 Fit Two-Gaussian baseline - fold 1

Inputs: hcsinhgoethe/ppg-cvae-prepared-fold01 and inputs/final/fold_01 via KRun --input.
CPU is sufficient. This stage does not require a completed cVAE job.
Fit a shared parameter bank using real train beats, fit seed 42 and at most 500 train windows.
Store the bank for generation seeds 42/43/44, plus 200-sample summaries at HR 60/80/120.
These are preliminary generation diagnostics; final test evaluation remains in step 07.


In [1]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEEDS = (42, 43, 44)
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


=== CHECKING /kaggle/input ===
Contents of /kaggle/input (1 items): ['datasets']
  Folder datasets contains 1 items
{'DATA_ROOT': '/kaggle/input/ppg-dalia-dataset', 'OUTPUT_ROOT': 'artifacts', 'device': 'cpu', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'torch': '2.11.0+cpu'}


In [2]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))

# Prepared dataset: https://www.kaggle.com/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01
import shutil
from zipfile import ZipFile
DATASET_SLUG = 'ppg-cvae-prepared-fold01'
ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
fold_name = f'fold_{FOLD:02d}'
required_files = ('train.npz', 'val.npz', 'manifest.csv', 'normalization_stats.json', 'preprocessing_config.json')
destination = OUTPUT_ROOT/'prepared'/fold_name

# Discover the mounted dataset rather than assuming Kaggle's directory layout.
if ARTIFACT_INPUT_ROOT:
    dataset_roots = [Path(ARTIFACT_INPUT_ROOT)]
else:
    dataset_roots = sorted(p for p in input_dir.rglob(DATASET_SLUG) if p.is_dir()) if input_dir.exists() else []
if not dataset_roots:
    raise FileNotFoundError(f'Attach hcsinhgoethe/{DATASET_SLUG} with krun --dataset, or set PPGCVAE_INPUT_ROOT.')

prepared_sources = []
for root in dataset_roots:
    if not root.is_dir(): raise FileNotFoundError(root)
    for train_file in root.rglob('train.npz'):
        candidate = train_file.parent
        if candidate.name.startswith('fold_') and candidate.name != fold_name: continue
        if all((candidate/name).is_file() for name in required_files):
            prepared_sources.append(candidate)
if len(prepared_sources) > 1:
    raise ValueError(f'Ambiguous prepared artifact directories: {prepared_sources}; set PPGCVAE_INPUT_ROOT explicitly.')
if prepared_sources:
    prepared_source = prepared_sources[0]
    if prepared_source.resolve() != destination.resolve():
        shutil.copytree(prepared_source, destination, dirs_exist_ok=True)
    print('Imported prepared arrays from:', prepared_source)
else:
    # Also support a dataset whose uploaded ZIP has not been unpacked by Kaggle.
    archives = []
    for root in dataset_roots:
        for archive in root.rglob('*.zip'):
            with ZipFile(archive) as z:
                prefix = f'prepared/{fold_name}/'
                if all(prefix+name in z.namelist() for name in required_files):
                    archives.append(archive)
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one prepared {fold_name} directory or ZIP in {dataset_roots}; found archives: {archives}')
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as z:
        prefix = f'prepared/{fold_name}/'
        for name in (*required_files, 'test.npz', 'subject_summary.csv', 'hr_coverage.csv'):
            member = prefix+name
            if member in z.namelist():
                with z.open(member) as src, (destination/name).open('wb') as dst:
                    shutil.copyfileobj(src, dst)
    print('Imported prepared arrays from ZIP:', archives[0])
assert all((destination/name).is_file() for name in required_files)
print('Prepared fold directory:', destination)

# Proxy is a differentiable estimate, not a ground-truth HR label.
# Preserve validation quality metadata in every tuning run for interpretation.
final_input = repo_root/'inputs'/'final'/fold_name
proxy_source = final_input/'proxy'
for name in ('hr_proxy.pt','detector_config.json','hr_proxy_validation_metrics.json','detector_validation_metrics.json'):
    if not (proxy_source/name).is_file():
        raise FileNotFoundError(f'Missing {proxy_source/name}; pass --input inputs/final/fold_01 to KRun.')
proxy_destination = OUTPUT_ROOT/'proxy'/fold_name
shutil.copytree(proxy_source,proxy_destination,dirs_exist_ok=True)
import json
proxy_quality = json.loads((proxy_destination/'hr_proxy_validation_metrics.json').read_text())
detector_quality = json.loads((proxy_destination/'detector_validation_metrics.json').read_text())
print('Proxy validation MAE bpm:',proxy_quality['best_val_mae_bpm'])
print('Detector real validation metrics:',detector_quality['selected'])

# Verify the transferred input package before using its selected configuration.
import hashlib
manifest = json.loads((final_input/'source_manifest.json').read_text(encoding='utf-8'))
assert manifest['fold'] == FOLD
for relative, expected in manifest['files_sha256'].items():
    actual = hashlib.sha256((final_input/relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Input checksum mismatch: {relative}')
selected_source = final_input/'tuning'/'selected_config.json'
selected = json.loads(selected_source.read_text(encoding='utf-8'))
if selected['fold'] != FOLD:
    raise ValueError('Selected configuration belongs to a different fold')
tuning_destination = OUTPUT_ROOT/'tuning'/fold_name
shutil.copytree(final_input/'tuning', tuning_destination, dirs_exist_ok=True)
print('Selected configuration:', selected)
print('Baseline generation seeds:', SEEDS)
print('Baseline fits train only; cVAE checkpoints are not required; test remains closed.')


FoldSubjects(fold_id=1, train_subjects=frozenset({'S9', 'S8', 'S11', 'S7', 'S10', 'S14', 'S15', 'S13', 'S12'}), val_subjects=frozenset({'S5', 'S4', 'S6'}), test_subjects=frozenset({'S2', 'S1', 'S3'}))


Imported prepared arrays from: /kaggle/input/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01/prepared/fold_01
Prepared fold directory: artifacts/prepared/fold_01
Proxy validation MAE bpm: 11.888657971351767
Detector real validation metrics: {'prominence': 0.1, 'V': 0.9999155191349159, 'mae_hr': 9.10020403735541, 'A3': 0.4336402804764721}
Selected configuration: {'fold': 1, 'beta': 0.01, 'lambda_hr': 0.1, 'supported_targets': [60, 80, 120], 'validation_seed': 1042, 'num_val_samples': 200, 'selection_rule': 'V>=95% at supported targets, MAE, 0.1 bpm tie RMSE; fallback A3', 'selected_metrics': {'beta': 0.01, 'lambda_hr': 0.1, 'all_V_ge_95': True, 'mean_mae_hr': 19.693561122192943, 'mean_A3': 0.07833333333333332, 'val_rmse': 0.5547592043876648}}
Baseline generation seeds: (42, 43, 44)
Baseline fits train only; cVAE checkpoints are not required; test remains closed.


In [3]:
import pandas as pd
import numpy as np
from ppg_cvae.workflow import fit_baseline, TARGETS
from ppg_cvae.baselines.gaussian import GaussianParams, synthesize_two_gaussian
from ppg_cvae.evaluation.heart_rate import generation_metrics
from ppg_cvae.evaluation.diversity import aligned_beat_diversity
from ppg_cvae.utils.io import save_json

# Fit once: all generation seeds use the same train-fitted parameter bank.
FIT_SEED = 42
MAX_TRAIN_WINDOWS = 500
NUM_DIAGNOSTIC_SAMPLES = 200
run_dir = fit_baseline(OUTPUT_ROOT, FOLD, FIT_SEED, max_train_windows=MAX_TRAIN_WINDOWS)
fitted = json.loads((run_dir/'gaussian_fit.json').read_text(encoding='utf-8'))
bank = [GaussianParams(**params) for params in fitted['parameters']]
assert fitted['fit_split'] == 'train' and fitted['num_fitted_beats'] > 0
with np.load(OUTPUT_ROOT/'prepared'/fold_name/'train.npz', allow_pickle=False) as prepared:
    if not np.isfinite(prepared['x']).all():
        raise ValueError('Non-finite train windows')
detector = json.loads((proxy_destination/'detector_config.json').read_text(encoding='utf-8'))
for seed in SEEDS:
    destination = OUTPUT_ROOT/'gaussian'/fold_name/f'seed_{seed}'
    destination.mkdir(parents=True, exist_ok=True)
    if destination != run_dir:
        shutil.copy2(run_dir/'gaussian_fit.json', destination/'gaussian_fit.json')
    summaries = []
    for hr in TARGETS:
        waves = synthesize_two_gaussian(hr, NUM_DIAGNOSTIC_SAMPLES, params=bank, seed=seed)
        _, summary = generation_metrics(waves, hr, **detector)
        summary.update(aligned_beat_diversity(waves, **detector))
        summary.update(fold=FOLD, seed=seed, method='gaussian', stage='preliminary_generation')
        summaries.append(summary)
    pd.DataFrame(summaries).to_csv(destination/'prior_validation_summary.csv', index=False)
    print(f'Seed {seed}:', flush=True)
    print(pd.DataFrame(summaries).to_string(index=False), flush=True)
save_json(dict(fold=FOLD, fit_seed=FIT_SEED, generation_seeds=list(SEEDS),
               max_train_windows=MAX_TRAIN_WINDOWS, num_fitted_beats=len(bank),
               num_diagnostic_samples=NUM_DIAGNOSTIC_SAMPLES, test_accessed=False,
               parameter_bank='shared across generation seeds',
               detector_source_job='20261004-050604-c55386'),
          OUTPUT_ROOT/'gaussian'/fold_name/'baseline_plan.json')
print('Saved train-fitted Gaussian bank for seeds 42/43/44. Final test evaluation remains in step 07.')


Seed 42:


 target_hr   mae_hr  median_abs_error    V    A3  failed_windows  num_samples  mean_temporal_std  num_beats  fold  seed   method                  stage
        60 8.034532               0.0 0.98 0.855               4          200           0.200624       1349     1    42 gaussian preliminary_generation
        80 1.616162               0.0 0.99 0.970               2          200           0.145749       1730     1    42 gaussian preliminary_generation
       120 0.000000               0.0 1.00 1.000               0          200           0.141703       2775     1    42 gaussian preliminary_generation


Seed 43:


 target_hr   mae_hr  median_abs_error     V    A3  failed_windows  num_samples  mean_temporal_std  num_beats  fold  seed   method                  stage
        60 6.570088               0.0 0.960 0.865               8          200           0.181384       1263     1    43 gaussian preliminary_generation
        80 2.447018               0.0 0.995 0.965               1          200           0.145067       1759     1    43 gaussian preliminary_generation
       120 0.000000               0.0 1.000 1.000               0          200           0.136920       2763     1    43 gaussian preliminary_generation


Seed 44:


 target_hr   mae_hr  median_abs_error     V    A3  failed_windows  num_samples  mean_temporal_std  num_beats  fold  seed   method                  stage
        60 7.683755               0.0 0.965 0.840               7          200           0.185323       1329     1    44 gaussian preliminary_generation
        80 2.800000               0.0 1.000 0.965               0          200           0.132332       1791     1    44 gaussian preliminary_generation
       120 0.000000               0.0 1.000 1.000               0          200           0.125717       2782     1    44 gaussian preliminary_generation


Saved train-fitted Gaussian bank for seeds 42/43/44. Final test evaluation remains in step 07.
